# Learning Activity: Training a Neural Network by Hand — The XOR Problem

**Student:** Elián David Martínez Orozco  
**Professor:** Dr. rer. nat. Humberto Llinás  
**Course:** Fundamentals of AI with Neural Networks and Transformers  
**Institution:** Universidad del Norte — Barranquilla, Colombia

---

## Reference

This activity is based on the class notes developed by the professor:

> Llinás, H. (2026). *AI Materials — Table of Contents*. RPubs. Available at: [https://rpubs.com/hllinas/R_AI1_toc](https://rpubs.com/hllinas/R_AI1_toc)

All mathematical notation, network architecture, initial parameters, and training procedure follow directly from those notes — specifically from Chapter 12: *Example: Manual Forward Pass for a Single XOR Observation*.

---

## Overview

The XOR (exclusive-OR) problem is a classical benchmark in neural networks. It cannot be solved by a single-layer perceptron because its classes are not linearly separable — no straight line can correctly partition the four input combinations. This activity demonstrates, step by step, how a small feedforward network with one hidden layer learns to approximate the XOR function through gradient descent and backpropagation.

| Property | Value |
|----------|-------|
| Architecture | 2 inputs → 2 hidden neurons (sigmoid) → 1 output (sigmoid) |
| Training observation | $x_1 = 0,\ x_2 = 1,\ y = 1$ |
| Learning rate | $\alpha = 0.25$ |
| Loss function | $E = \frac{1}{2}(y - \hat{y})^2$ |

> **Constraint:** No neural network libraries or automatic differentiation tools are used. All forward passes, backpropagation steps, and parameter updates are implemented from scratch using only NumPy.

---
## Setup — Imports and Global Definitions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Core functions (from scratch — no ML libraries) ───────────────────────────
def sigmoid(x):
    """Logistic sigmoid activation: sigma(x) = 1 / (1 + exp(-x))."""
    return 1.0 / (1.0 + np.exp(-x))

print(f"numpy      : {np.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"matplotlib : {plt.matplotlib.__version__}")
print("Setup complete ✓")

---
## Network Architecture Diagram

The diagram below reproduces the architecture from **Figure 12.1** in the class notes (Llinás, 2026). It shows the exact network used throughout this activity:

- **Input layer** $(x_1, x_2)$: receives the two binary inputs
- **Hidden layer** $(h_1, h_2)$: each neuron computes $h_j = \sigma(z_j)$, where $z_j$ is its weighted pre-activation
- **Output layer** $(\hat{y})$: produces the final prediction using the same sigmoid function
- **Bias nodes** (gray triangles labeled "1"): each contributes a learnable offset $b_j$ through green arrows
- **Weights** $w_1, \ldots, w_6$: the six learnable parameters connecting input→hidden and hidden→output

Understanding this diagram is essential: every computation in the following sections maps directly to a specific node, arrow, or weight shown here.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# -----------------------------
# Positions
# -----------------------------
pos = {
    "x1": (0, 2),
    "x2": (0, 0),
    "h1": (3, 2),
    "h2": (3, 0),
    "y":  (6, 1),
    "b_hidden": (1.6, 4.5),
    "b_output": (4.2, 4.5)
}

node_radius = 0.35

# -----------------------------
# Draw nodes
# -----------------------------
for key in ["x1", "x2", "h1", "h2", "y"]:
    x, y = pos[key]
    circle = plt.Circle((x, y), node_radius, color='crimson')
    ax.add_patch(circle)

# -----------------------------
# Draw bias triangles
# -----------------------------
def draw_triangle(center):
    x, y = center
    triangle = plt.Polygon([
        (x, y),
        (x - 0.28, y - 0.50),
        (x + 0.28, y - 0.50)
    ], color='gray')
    ax.add_patch(triangle)
    ax.text(x, y - 0.30, "1", ha='center', va='center',
            fontsize=12, color='black')

draw_triangle(pos["b_hidden"])
draw_triangle(pos["b_output"])

# -----------------------------
# Helper: shorten arrows
# -----------------------------
def shorten(p1, p2, start_offset=0.36, end_offset=0.42):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    v = p2 - p1
    d = np.linalg.norm(v)
    u = v / d
    return p1 + start_offset * u, p2 - end_offset * u

def arrow(p1, p2, color='black', lw=1.5):
    if color == 'green':
        q1, q2 = shorten(p1, p2, start_offset=0, end_offset=0.50)
        mscale = 25
        width = 1
    else:
        q1, q2 = shorten(p1, p2, start_offset=0.36, end_offset=0.42)
        mscale = 18
        width = lw

    ax.annotate(
        "",
        xy=q2, xytext=q1,
        arrowprops=dict(
            arrowstyle="-|>",
            color=color,
            lw=width,
            mutation_scale=mscale,
            shrinkA=0,
            shrinkB=0
        )
    )

# -----------------------------
# Connections
# -----------------------------
b_hidden_base = (pos["b_hidden"][0], pos["b_hidden"][1] - 0.50)
b_output_base = (pos["b_output"][0], pos["b_output"][1] - 0.50)

# Black arrows (weights)
arrow(pos["x1"], pos["h1"])
arrow(pos["x1"], pos["h2"])
arrow(pos["x2"], pos["h1"])
arrow(pos["x2"], pos["h2"])
arrow(pos["h1"], pos["y"])
arrow(pos["h2"], pos["y"])

# Green arrows (biases)
arrow(b_hidden_base, pos["h1"], color='green')
arrow(b_hidden_base, pos["h2"], color='green')
arrow(b_output_base, pos["y"],  color='green')

# -----------------------------
# Node labels
# -----------------------------
ax.text(*pos["x1"], r"$X_1$", ha='center', va='center', fontsize=18, fontweight='bold')
ax.text(*pos["x2"], r"$X_2$", ha='center', va='center', fontsize=18, fontweight='bold')
ax.text(*pos["y"],  r"$\hat{y}$", ha='center', va='center', fontsize=18, fontweight='bold')
ax.text(pos["y"][0], pos["y"][1] - 0.6, r"$y$", ha='center', va='center', fontsize=18,
        bbox=dict(facecolor='#FFF59D', edgecolor='none', boxstyle='round,pad=0.3'))
ax.text(*pos["h1"], r"$h_1$", ha='center', va='center', fontsize=18, fontweight='bold')
ax.text(*pos["h2"], r"$h_2$", ha='center', va='center', fontsize=18, fontweight='bold')

# -----------------------------
# Weight labels
# -----------------------------
ax.text(1.45,  2.1,  r"$w_1$", fontsize=15, fontweight='bold')
ax.text(0.50,  1.22, r"$w_2$", fontsize=15, fontweight='bold')
ax.text(0.68,  0.7,  r"$w_3$", fontsize=15, fontweight='bold')
ax.text(1.45, -0.2,  r"$w_4$", fontsize=15, fontweight='bold')
ax.text(4.15,  1.70, r"$w_5$", fontsize=15, fontweight='bold')
ax.text(4.15,  0.12, r"$w_6$", fontsize=15, fontweight='bold')

# -----------------------------
# Bias labels
# -----------------------------
ax.text(2.4,  3.3,  r"$b_1$", color='green', fontsize=15, fontweight='bold')
ax.text(2.8,  1.0,  r"$b_2$", color='green', fontsize=15, fontweight='bold')
ax.text(5.05, 2.92, r"$b_3$", color='green', fontsize=15, fontweight='bold')

# -----------------------------
# Layer labels
# -----------------------------
ax.text(0, -1, r"Input Layer $\in \mathbb{R}^2$",  ha='center', fontsize=15, color='gray')
ax.text(3, -1, r"Hidden Layer $\in \mathbb{R}^2$", ha='center', fontsize=15, color='gray')
ax.text(6, -1, r"Output Layer $\in \mathbb{R}^1$", ha='center', fontsize=15, color='gray')

ax.set_xlim(-1, 7)
ax.set_ylim(-1.5, 4.5)
ax.axis('off')

plt.tight_layout()
plt.show()

**Reading the diagram.** Each input neuron fans out to both hidden neurons, contributing through weights $w_1, w_2$ (from $x_1$) and $w_3, w_4$ (from $x_2$). The hidden neurons combine these signals — plus the bias offset $b_1$ or $b_2$ — and pass the result through a sigmoid. Their outputs $h_1, h_2$ feed the output neuron through weights $w_5, w_6$, with one more bias term $b_3$. The yellow label $y$ below $\hat{y}$ reminds us that the target value is always available during training for computing the error.

A key architectural observation: **information only flows left to right during the forward pass** and right to left during backpropagation. This directionality is what makes the chain rule the correct tool for computing gradients.

---
## Section 1 — Continuation of the Manual XOR Example (Third Epoch)

Chapters 12.0.6–12.0.8 of the class notes (Llinás, 2026) completed two full epochs of manual training for the observation $(x_1, x_2, y) = (0, 1, 1)$. We continue from the parameters obtained at the end of Epoch 2:

$$w_1 = 0.10000,\quad w_2 = 0.50000,\quad w_3 = -0.69764,\quad w_4 = 0.30517,\quad w_5 = 0.21724,\quad w_6 = 0.42985$$
$$b_1 = b_2 = b_3 = 0$$

---
### Forward Pass

**Hidden neuron $h_1$:**
$$z_1 = w_1 x_1 + w_3 x_2 + b_1 = (0.1)(0) + (-0.69764)(1) + 0 = -0.69764$$
$$h_1 = \sigma(-0.69764) = \frac{1}{1+e^{0.69764}} = 0.33234$$

**Hidden neuron $h_2$:**
$$z_2 = w_2 x_1 + w_4 x_2 + b_2 = (0.5)(0) + (0.30517)(1) + 0 = 0.30517$$
$$h_2 = \sigma(0.30517) = \frac{1}{1+e^{-0.30517}} = 0.57571$$

**Output neuron $\hat{y}$:**
$$z_3 = w_5 h_1 + w_6 h_2 + b_3 = (0.21724)(0.33234) + (0.42985)(0.57571) + 0 = 0.31966$$
$$\hat{y} = \sigma(0.31966) = 0.57924, \qquad E = \tfrac{1}{2}(1 - 0.57924)^2 = 0.08852$$

---
### Backpropagation

$$\frac{\partial E}{\partial z_3} = (\hat{y}-y)\cdot\hat{y}(1-\hat{y}) = (0.57924-1)(0.57924)(0.42076) = -0.10255$$

$$\frac{\partial E}{\partial w_5} = (-0.10255)(0.33234) = -0.03408 \qquad \frac{\partial E}{\partial w_6} = (-0.10255)(0.57571) = -0.05904$$

$$\frac{\partial E}{\partial z_1} = (-0.10255)(0.21724)(0.33234)(0.66766) = -0.00494$$
$$\frac{\partial E}{\partial z_2} = (-0.10255)(0.42985)(0.57571)(0.42429) = -0.01077$$

$$\frac{\partial E}{\partial w_3} = (-0.00494)(1) = -0.00494, \quad \frac{\partial E}{\partial w_4} = (-0.01077)(1) = -0.01077$$
$$\frac{\partial E}{\partial w_1} = (-0.00494)(0) = 0, \quad \frac{\partial E}{\partial w_2} = (-0.01077)(0) = 0$$

---
### Updated Parameters after Epoch 3

| | $w_1$ | $w_2$ | $w_3$ | $w_4$ | $w_5$ | $w_6$ |
|-|-------|-------|-------|-------|-------|-------|
| **Before** | 0.10000 | 0.50000 | −0.69764 | 0.30517 | 0.21724 | 0.42985 |
| **Gradient** | 0 | 0 | −0.00494 | −0.01077 | −0.03408 | −0.05904 |
| **After** | 0.10000 | 0.50000 | −0.69640 | 0.30786 | 0.22576 | 0.44461 |

In [ ]:
# ── Epoch 3 — numerical verification ─────────────────────────────────────────
w1, w2, w3, w4, w5, w6 = 0.10000, 0.50000, -0.69764, 0.30517, 0.21724, 0.42985
b1, b2, b3 = 0.0, 0.0, 0.0
alpha = 0.25
x1_obs, x2_obs, y_obs = 0.0, 1.0, 1.0

# Forward pass
z1   = w1*x1_obs + w3*x2_obs + b1
h1   = sigmoid(z1)
z2   = w2*x1_obs + w4*x2_obs + b2
h2   = sigmoid(z2)
z3   = w5*h1 + w6*h2 + b3
yhat = sigmoid(z3)
loss = 0.5*(y_obs - yhat)**2

# Backpropagation
dE_dz3 = (yhat - y_obs) * yhat * (1 - yhat)
dE_dw5 = dE_dz3 * h1;        dE_dw6 = dE_dz3 * h2
dE_dz1 = dE_dz3 * w5 * h1 * (1 - h1)
dE_dz2 = dE_dz3 * w6 * h2 * (1 - h2)
dE_dw1 = dE_dz1 * x1_obs;    dE_dw3 = dE_dz1 * x2_obs
dE_dw2 = dE_dz2 * x1_obs;    dE_dw4 = dE_dz2 * x2_obs

# Updates
w1n = w1 - alpha*dE_dw1;  w2n = w2 - alpha*dE_dw2
w3n = w3 - alpha*dE_dw3;  w4n = w4 - alpha*dE_dw4
w5n = w5 - alpha*dE_dw5;  w6n = w6 - alpha*dE_dw6

print("EPOCH 3 — Full computation")
print("=" * 50)
print(f"  z1={z1:.5f}   h1={h1:.5f}")
print(f"  z2={z2:.5f}   h2={h2:.5f}")
print(f"  z3={z3:.5f}   ŷ ={yhat:.5f}")
print(f"  E ={loss:.5f}   |err|={abs(y_obs-yhat):.5f}")
print()
print(f"  ∂E/∂z3={dE_dz3:.5f}")
print(f"  ∂E/∂w5={dE_dw5:.5f}   ∂E/∂w6={dE_dw6:.5f}")
print(f"  ∂E/∂z1={dE_dz1:.5f}   ∂E/∂z2={dE_dz2:.5f}")
print(f"  ∂E/∂w3={dE_dw3:.5f}   ∂E/∂w4={dE_dw4:.5f}")
print()
print("  Updated weights:")
print(f"  w3={w3n:.5f}  w4={w4n:.5f}  w5={w5n:.5f}  w6={w6n:.5f}")
print()
print("  Cross-epoch comparison:")
print(f"  Epoch 1: ŷ=0.57350  E=0.09095  |err|=0.42650")
print(f"  Epoch 2: ŷ=0.57638  E=0.08973  |err|=0.42362  (ΔE = -0.00122)")
print(f"  Epoch 3: ŷ={yhat:.5f}  E={loss:.5f}  |err|={abs(y_obs-yhat):.5f}  (ΔE = {loss-0.08973:.5f})")

**Reflection.** Three observations stand out from this third epoch.

First, the loss decreases by a nearly constant amount at each epoch ($\approx -0.001$), which is characteristic of gradient descent on a smooth convex surface: progress is regular but slow when the learning rate is small relative to the curvature.

Second, the weights $w_1$ and $w_2$ receive a gradient of exactly zero because the input $x_1 = 0$ multiplies both. This is not a flaw — it is a direct consequence of the chain rule. The error signal propagated back to the hidden layer is then multiplied by the pre-activation input, so if that input is zero, no information reaches those weights. This highlights an important practical issue: training on a single observation with a zero feature leaves part of the network frozen.

Third, the bias terms $b_1, b_2, b_3$ also remain at zero across all three epochs because their gradients ($\partial E/\partial z_j$ for each layer) are also affected by this structural limitation. In a richer training setting with all four XOR observations, biases would update freely and contribute significantly to convergence.

---
## Section 2 — Automated Training for 15 Epochs

We now automate the same procedure over 15 epochs, starting from the **original initial weights** defined in Figure 12.2 of the class notes (Llinás, 2026). All computations are performed from scratch — no ML library functions are invoked at any point.

In [ ]:
# ── Automated training — 15 epochs (from scratch) ────────────────────────────
W = dict(w1=0.1, w2=0.5, w3=-0.7, w4=0.3, w5=0.2, w6=0.4)
B = dict(b1=0.0, b2=0.0, b3=0.0)
x1_obs, x2_obs, y_obs = 0.0, 1.0, 1.0
alpha = 0.25
history = []

for ep in range(1, 16):
    # Forward
    z1   = W["w1"]*x1_obs + W["w3"]*x2_obs + B["b1"]
    h1   = sigmoid(z1)
    z2   = W["w2"]*x1_obs + W["w4"]*x2_obs + B["b2"]
    h2   = sigmoid(z2)
    z3   = W["w5"]*h1 + W["w6"]*h2 + B["b3"]
    yhat = sigmoid(z3)
    loss = 0.5*(y_obs - yhat)**2

    # Backprop
    dE_dz3 = (yhat - y_obs) * yhat * (1 - yhat)
    dE_dw5 = dE_dz3 * h1;        dE_dw6 = dE_dz3 * h2
    dE_dz1 = dE_dz3 * W["w5"] * h1 * (1 - h1)
    dE_dz2 = dE_dz3 * W["w6"] * h2 * (1 - h2)
    dE_dw1 = dE_dz1 * x1_obs;    dE_dw3 = dE_dz1 * x2_obs
    dE_dw2 = dE_dz2 * x1_obs;    dE_dw4 = dE_dz2 * x2_obs

    # Update
    for k, g in [("w1",dE_dw1),("w2",dE_dw2),("w3",dE_dw3),
                 ("w4",dE_dw4),("w5",dE_dw5),("w6",dE_dw6)]:
        W[k] -= alpha * g

    history.append(dict(
        epoch=ep, z1=z1, z2=z2, z3=z3, h1=h1, h2=h2,
        yhat=yhat, y=y_obs, abs_err=abs(y_obs-yhat), loss=loss,
        grad_z3=dE_dz3,
        w1=W["w1"],w2=W["w2"],w3=W["w3"],
        w4=W["w4"],w5=W["w5"],w6=W["w6"]
    ))

df = pd.DataFrame(history)

# ── Table 1: forward metrics ──────────────────────────────────────────────────
print("Table 1 — Forward pass and loss metrics")
print("=" * 88)
print(f"{'Ep':>3} {'z1':>9} {'z2':>9} {'z3':>9} {'h1':>8} {'h2':>8} "
      f"{'ŷ':>8} {'y':>4} {'|err|':>8} {'Loss':>9} {'∂E/∂z3':>9}")
print("-" * 88)
for _, r in df.iterrows():
    print(f"{int(r.epoch):>3} {r.z1:>9.5f} {r.z2:>9.5f} {r.z3:>9.5f} "
          f"{r.h1:>8.5f} {r.h2:>8.5f} {r.yhat:>8.5f} {int(r.y):>4} "
          f"{r.abs_err:>8.5f} {r.loss:>9.5f} {r.grad_z3:>9.5f}")

In [ ]:
# ── Table 2: weight evolution ─────────────────────────────────────────────────
print("Table 2 — Weight evolution per epoch")
print("=" * 72)
print(f"{'Ep':>3} {'w1':>9} {'w2':>9} {'w3':>10} {'w4':>9} {'w5':>9} {'w6':>9}")
print("-" * 72)
for _, r in df.iterrows():
    print(f"{int(r.epoch):>3} {r.w1:>9.5f} {r.w2:>9.5f} {r.w3:>10.5f} "
          f"{r.w4:>9.5f} {r.w5:>9.5f} {r.w6:>9.5f}")

**Reflection.** Several structural patterns are visible across these 15 epochs.

**Frozen weights.** The columns $w_1$ and $w_2$ remain at their initial values in every epoch. This is not an accident: since $x_1 = 0$, the gradient $\partial E / \partial w_1 = (\partial E/\partial z_1) \cdot x_1 = 0$ identically. From the network's perspective, it has never "seen" evidence about what $w_1$ and $w_2$ should be — the input feature $x_1$ was always zero, so those connections never contributed to any prediction.

**Monotone improvement.** Both the loss and the absolute error decrease in every epoch. This is guaranteed by the gradient descent update rule as long as the learning rate $\alpha$ is small enough (here $\alpha = 0.25$ is conservative). The network is consistently moving in the right direction.

**Slowing gradient.** The column $\partial E / \partial z_3$ decreases in magnitude each epoch. As $\hat{y}$ approaches $y = 1$, the error $(\hat{y} - y)$ shrinks, and the term $\hat{y}(1-\hat{y})$ also decreases as the sigmoid approaches saturation. This is a fundamental characteristic of sigmoid-based training: updates slow down as the network improves, which can require many more epochs to reach convergence.

---
## Section 3 — Graphical Analysis of the Training Process

In [ ]:
# ── 4-panel training analysis ─────────────────────────────────────────────────
epochs = df["epoch"].values
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(
    "Training Process Analysis — Single Observation XOR\n"
    r"$(x_1=0,\ x_2=1,\ y=1)$,  $\alpha=0.25$,  15 epochs",
    fontsize=13, fontweight="bold"
)

# (a) Loss
ax = axes[0, 0]
ax.plot(epochs, df["loss"], "o-", color="#C8102E", lw=2, ms=5)
ax.fill_between(epochs, df["loss"], alpha=0.12, color="#C8102E")
ax.set_title("(a) Loss $E$ across epochs", fontsize=11)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss $E$")
ax.grid(alpha=0.3)
ax.annotate(f"Ep 1: {df.loss.iloc[0]:.5f}",
            xy=(1, df.loss.iloc[0]), xytext=(2, df.loss.iloc[0]+0.0015),
            fontsize=8, color="#C8102E")
ax.annotate(f"Ep 15: {df.loss.iloc[-1]:.5f}",
            xy=(15, df.loss.iloc[-1]), xytext=(11, df.loss.iloc[-1]+0.0015),
            fontsize=8, color="#C8102E")

# (b) Absolute error
ax = axes[0, 1]
ax.plot(epochs, df["abs_err"], "s-", color="#1a3e8a", lw=2, ms=5)
ax.fill_between(epochs, df["abs_err"], alpha=0.12, color="#1a3e8a")
ax.set_title(r"(b) Absolute error $|y - \hat{y}|$", fontsize=11)
ax.set_xlabel("Epoch"); ax.set_ylabel(r"$|y - \hat{y}|$")
ax.grid(alpha=0.3)

# (c) Gradient magnitude
ax = axes[1, 0]
ax.plot(epochs, df["grad_z3"].abs(), "^-", color="#7c3aed", lw=2, ms=5)
ax.fill_between(epochs, df["grad_z3"].abs(), alpha=0.12, color="#7c3aed")
ax.set_title(r"(c) Output gradient magnitude $|\partial E/\partial z_3|$", fontsize=11)
ax.set_xlabel("Epoch"); ax.set_ylabel(r"$|\partial E/\partial z_3|$")
ax.grid(alpha=0.3)

# (d) Active weight evolution
ax = axes[1, 1]
color_map = {"w3":"#15803d", "w4":"#ea580c", "w5":"#7c3aed", "w6":"#0369a1"}
for wname, color in color_map.items():
    ax.plot(epochs, df[wname], "o-", color=color, lw=1.8, ms=4, label=f"${wname}$")
ax.axhline(0, color="gray", lw=0.7, ls="--")
ax.set_title("(d) Evolution of active weights", fontsize=11)
ax.set_xlabel("Epoch"); ax.set_ylabel("Weight value")
ax.legend(fontsize=9, ncol=2); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Reflection.** The four panels tell a coherent story about learning.

**Panel (a) — Loss.** The curve is smooth and monotonically decreasing, confirming that gradient descent is working correctly. The total reduction over 15 epochs is approximately 0.017 (from 0.09095 to 0.07520), which is modest — indicating that many more epochs would be needed for the prediction to get close to the target $y = 1$.

**Panel (b) — Absolute error.** The error mirrors the loss curve exactly in shape, which is expected since $E = \frac{1}{2}|e|^2$ implies that $|e| = \sqrt{2E}$. Both panels confirm that the network is improving its prediction consistently.

**Panel (c) — Gradient.** The gradient $|\partial E/\partial z_3|$ decreases as the prediction improves. This illustrates the *vanishing gradient* phenomenon in its mild form: the error signal used to update weights becomes weaker as the network approaches the correct output. The sigmoid activation is particularly prone to this because its derivative $\sigma(z)(1-\sigma(z))$ approaches zero as $|z|$ grows.

**Panel (d) — Weights.** Only $w_3, w_4, w_5, w_6$ update because they are the only weights whose gradient paths include a non-zero input. $w_5$ and $w_6$ (output layer) update faster than $w_3, w_4$ (hidden layer), because their gradient does not require the extra chain-rule multiplication through the hidden activation derivative. This is the signature of the vanishing gradient through layers.

---
## Section 4 — Sensitivity to Initial Parameters

We repeat the 15-epoch training with an alternative initialization that includes **non-zero biases** and **mixed-sign weights**, and compare the two experiments.

In [ ]:
# ── Alternative initialization ────────────────────────────────────────────────
W_alt = dict(w1=0.6, w2=-0.4, w3=0.8, w4=-0.3, w5=0.5, w6=-0.6)
B_alt = dict(b1=0.1, b2=-0.1, b3=0.05)
history_alt = []

for ep in range(1, 16):
    z1   = W_alt["w1"]*x1_obs + W_alt["w3"]*x2_obs + B_alt["b1"]
    h1   = sigmoid(z1)
    z2   = W_alt["w2"]*x1_obs + W_alt["w4"]*x2_obs + B_alt["b2"]
    h2   = sigmoid(z2)
    z3   = W_alt["w5"]*h1 + W_alt["w6"]*h2 + B_alt["b3"]
    yhat = sigmoid(z3)
    loss = 0.5*(y_obs - yhat)**2

    dE_dz3 = (yhat - y_obs) * yhat * (1 - yhat)
    dE_dw5 = dE_dz3*h1;           dE_dw6 = dE_dz3*h2
    dE_db3 = dE_dz3
    dE_dz1 = dE_dz3 * W_alt["w5"] * h1 * (1-h1)
    dE_dz2 = dE_dz3 * W_alt["w6"] * h2 * (1-h2)
    dE_dw3 = dE_dz1*x2_obs;       dE_dw4 = dE_dz2*x2_obs
    dE_db1 = dE_dz1;               dE_db2 = dE_dz2

    W_alt["w3"] -= alpha*dE_dw3;  W_alt["w4"] -= alpha*dE_dw4
    W_alt["w5"] -= alpha*dE_dw5;  W_alt["w6"] -= alpha*dE_dw6
    B_alt["b1"] -= alpha*dE_db1;  B_alt["b2"] -= alpha*dE_db2
    B_alt["b3"] -= alpha*dE_db3

    history_alt.append(dict(
        epoch=ep, yhat=yhat, loss=loss,
        abs_err=abs(y_obs-yhat), grad_z3=dE_dz3
    ))

df_alt = pd.DataFrame(history_alt)

# ── Comparison plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle("Sensitivity to Initial Parameters — Original vs Alternative",
             fontsize=12, fontweight="bold")

for ax, (col_orig, col_alt, title, ylabel) in zip(axes, [
    ("loss",     "loss",     "(a) Loss",           "Loss $E$"),
    ("abs_err",  "abs_err",  "(b) Absolute error", r"$|y-\hat{y}|$"),
    ("grad_z3",  "grad_z3",  "(c) Gradient magnitude", r"$|\partial E/\partial z_3|$"),
]):
    ax.plot(epochs, df[col_orig].abs(), "o-", color="#C8102E", lw=2, ms=5,
            label="Original init.")
    ax.plot(df_alt["epoch"], df_alt[col_alt].abs(), "s--", color="#1a3e8a",
            lw=2, ms=5, label="Alt. init.")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"{'Metric':<32} {'Original':>12} {'Alternative':>13}")
print("-"*58)
print(f"{'Initial ŷ (Ep 1)':<32} {df.yhat.iloc[0]:>12.5f} {df_alt.yhat.iloc[0]:>13.5f}")
print(f"{'Initial loss':<32} {df.loss.iloc[0]:>12.5f} {df_alt.loss.iloc[0]:>13.5f}")
print(f"{'Final ŷ (Ep 15)':<32} {df.yhat.iloc[-1]:>12.5f} {df_alt.yhat.iloc[-1]:>13.5f}")
print(f"{'Final loss':<32} {df.loss.iloc[-1]:>12.5f} {df_alt.loss.iloc[-1]:>13.5f}")
print(f"{'Total loss reduction':<32} {df.loss.iloc[0]-df.loss.iloc[-1]:>12.5f} {df_alt.loss.iloc[0]-df_alt.loss.iloc[-1]:>13.5f}")

**Reflection.** The comparison reveals how sensitive gradient descent is to the starting point.

The **original initialization** begins with all biases at zero and small, same-sign weights. This places the network in a region of the loss surface where the gradients are moderate and updates are small. Progress is steady but slow.

The **alternative initialization** introduces non-zero biases ($b_1 = 0.1$, $b_2 = -0.1$, $b_3 = 0.05$) and mixed-sign weights, including a negative output weight $w_6 = -0.6$. This starts the network in a different region of the loss surface. Since biases update in every epoch (their gradient does not depend on the input), the alternative initialization allows $b_1, b_2, b_3$ to actively shape the pre-activation signals throughout training.

Neither initialization converges to near-zero loss in 15 epochs with a single observation. The fundamental limitation is not the initialization but the training data: one XOR observation with $x_1 = 0$ leaves half the weight connections without any gradient signal, regardless of where we start.

---
## Section 5 — Training with Two XOR Observations (Vectorized)

We extend training to two observations — $(0,1)\mapsto 1$ and $(1,0)\mapsto 1$ — using a **matrix-based** forward and backward pass. This mirrors the general framework described in Llinás (2026) for multi-observation training.

$$X = \begin{pmatrix}0 & 1 \\ 1 & 0\end{pmatrix} \in \mathbb{R}^{2\times2}, \qquad y = \begin{pmatrix}1 \\ 1\end{pmatrix} \in \mathbb{R}^{2\times1}$$

**Matrix forward pass:**
$$Z^{(1)} = X\,W^{(0,1)} + \mathbf{1}\,{b^{(1)}}^\top \in \mathbb{R}^{2\times2}, \quad H^{(1)} = \sigma(Z^{(1)})$$
$$Z^{(2)} = H^{(1)}\,W^{(1,2)} + b^{(2)} \in \mathbb{R}^{2\times1}, \quad \hat{Y} = \sigma(Z^{(2)})$$

In [ ]:
# ── Two-observation vectorized training ───────────────────────────────────────
X2   = np.array([[0.,1.],[1.,0.]])
Y2   = np.array([[1.],[1.]])
# W01: shape (2,2) — rows=input neurons, cols=hidden neurons
W01  = np.array([[0.1, 0.5],[-0.7, 0.3]])
W12  = np.array([[0.2],[0.4]])
b1v  = np.zeros((1,2))
b2v  = np.zeros((1,1))
hist2 = []

for ep in range(1, 16):
    # Forward (matrix)
    Z1    = X2 @ W01 + b1v          # (2,2)
    H1    = sigmoid(Z1)              # (2,2)
    Z2    = H1 @ W12 + b2v           # (2,1)
    Yhat  = sigmoid(Z2)              # (2,1)
    loss  = float(np.mean(0.5*(Y2 - Yhat)**2))

    # Backward (matrix)
    delta2 = (Yhat - Y2) * Yhat * (1-Yhat)   # (2,1)
    dW12   = H1.T @ delta2                    # (2,1)
    db2    = delta2.mean(axis=0, keepdims=True)
    delta1 = (delta2 @ W12.T) * H1*(1-H1)    # (2,2)
    dW01   = X2.T @ delta1                   # (2,2)
    db1    = delta1.mean(axis=0, keepdims=True)

    # Update
    W01 -= alpha*dW01;  W12 -= alpha*dW12
    b1v -= alpha*db1;   b2v -= alpha*db2

    hist2.append(dict(
        epoch=ep,
        yhat1=float(Yhat[0,0]), yhat2=float(Yhat[1,0]),
        loss=loss, abs_err=float(np.mean(np.abs(Y2-Yhat))),
        grad_norm=float(np.linalg.norm(delta2))
    ))

df2 = pd.DataFrame(hist2)

print("Two-observation training — 15 epochs")
print(f"X shape: {X2.shape}   Y shape: {Y2.shape}")
print()
print(f"{'Ep':>3} {'ŷ₁':>8} {'ŷ₂':>8} {'Mean Loss':>11} {'Mean |err|':>11} {'‖δ²‖':>8}")
print("-"*55)
for _,r in df2.iterrows():
    print(f"{int(r.epoch):>3} {r.yhat1:>8.5f} {r.yhat2:>8.5f} "
          f"{r.loss:>11.5f} {r.abs_err:>11.5f} {r.grad_norm:>8.5f}")

In [ ]:
# ── 1-obs vs 2-obs comparison ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle("Two Observations vs Single Observation — Comparison",
             fontsize=12, fontweight="bold")

for ax, (c1, c2, title, ylabel) in zip(axes, [
    ("loss",     "loss",       "(a) Loss",           "Loss $E$"),
    ("abs_err",  "abs_err",    "(b) Absolute error", r"$|y-\hat{y}|$"),
    ("grad_z3",  "grad_norm",  "(c) Gradient norm",  "Gradient norm"),
]):
    ax.plot(epochs, df[c1].abs(), "o-", color="#C8102E", lw=2, ms=5, label="1 obs.")
    ax.plot(df2["epoch"], df2[c2].abs(), "s--", color="#1a6e2e", lw=2, ms=5, label="2 obs.")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Reflection.** The matrix formulation reveals the elegance of vectorized backpropagation. Instead of running two separate scalar loops, the entire mini-batch of two observations is processed simultaneously:

- In the **forward pass**, the matrix $X \in \mathbb{R}^{2 \times 2}$ produces two rows of pre-activations $Z^{(1)} \in \mathbb{R}^{2 \times 2}$ in one matrix multiplication.
- In the **backward pass**, the delta matrix $\delta^{(2)} \in \mathbb{R}^{2 \times 1}$ captures the error signal for both observations at once, and $\frac{\partial E}{\partial W^{(1,2)}} = H^{(1)\top} \delta^{(2)}$ accumulates both gradient contributions through a single multiplication.

The key difference from the single-observation experiment is that **both observations activate different input connections**. The first observation $(0,1)$ contributes gradients through $x_2 = 1$ and the second $(1,0)$ contributes through $x_1 = 1$. This means that for the first time, $w_1$ and $w_2$ receive non-zero gradients and begin updating — something that never happened in the single-observation case. This is a direct consequence of having more diverse training data.

---
## Section 6 — Interpretation and Comparison

In [ ]:
# ── Summary comparison — three experiments ────────────────────────────────────
print(f"{'Metric':<35} {'1-obs orig':>12} {'1-obs alt':>12} {'2-obs':>10}")
print("=" * 72)
rows = [
    ("Initial ŷ",
     df.yhat.iloc[0],     df_alt.yhat.iloc[0],     df2.yhat1.iloc[0]),
    ("Initial loss",
     df.loss.iloc[0],     df_alt.loss.iloc[0],     df2.loss.iloc[0]),
    ("Final ŷ (Ep 15)",
     df.yhat.iloc[-1],    df_alt.yhat.iloc[-1],    df2.yhat1.iloc[-1]),
    ("Final loss",
     df.loss.iloc[-1],    df_alt.loss.iloc[-1],    df2.loss.iloc[-1]),
    ("Total loss reduction",
     df.loss.iloc[0]  - df.loss.iloc[-1],
     df_alt.loss.iloc[0]- df_alt.loss.iloc[-1],
     df2.loss.iloc[0] - df2.loss.iloc[-1]),
    ("Final |error|",
     df.abs_err.iloc[-1], df_alt.abs_err.iloc[-1], df2.abs_err.iloc[-1]),
]
for name, v1, v2, v3 in rows:
    print(f"  {name:<33} {v1:>12.5f} {v2:>12.5f} {v3:>10.5f}")

**Reflection.** Comparing the three experiments makes several principles concrete:

**Why XOR requires a hidden layer.** As argued geometrically in Llinás (2026) and in Figure 11.1 of the class notes, no linear boundary can separate the XOR classes. Without a hidden layer, the network output is $\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + b)$ — a single sigmoid applied to a linear combination. This can only produce a linear decision boundary, which is insufficient. The hidden layer transforms the input space: $h_1$ and $h_2$ each define soft halfplanes, and their combination creates the nonlinear partition needed to separate $(0,0),(1,1)$ from $(0,1),(1,0)$.

**Effect of the number of observations.** With one observation, only two of the six weights receive gradients. With two complementary observations ($(0,1)$ and $(1,0)$), all six weights can update. This produces a richer gradient signal and faster convergence. The general principle is that the diversity of the training set — not just its size — determines which parameters can be learned.

**Effect of initialization.** The alternative initialization places the network at a different starting loss. Having non-zero biases allows the bias parameters to update and contribute to the optimization, even with limited input diversity. This does not solve the fundamental problem (single observation, frozen $w_1, w_2$), but it provides a more active gradient landscape.

**Limitations.** None of the three experiments produces a well-trained model. The fundamental bottleneck is training with a subset of the XOR table: the model cannot learn the full XOR function without seeing all four combinations.

---
## Section 7 — Conceptual Reflection

### How the network "learns from its mistakes"

Each iteration of training follows a cycle of five stages:

1. **Prediction ($\hat{y}$).** The network computes a prediction by propagating the input forward through two layers of linear transformations and sigmoid activations. At the start, this prediction is essentially arbitrary — determined by the initial weights, not by any knowledge of the target.

2. **Target ($y$) and error.** The target value $y = 1$ is known. The error $y - \hat{y}$ quantifies how far the current prediction is from the correct answer. In the first epoch, $\hat{y} = 0.57350$ and the error is $1 - 0.57350 = 0.42650$ — the network is significantly underestimating the XOR output.

3. **Loss function.** The loss $E = \frac{1}{2}(y-\hat{y})^2$ converts the error into a scalar that is smooth, differentiable, and bounded below by zero. The factor $\frac{1}{2}$ simplifies the derivative: $\partial E / \partial \hat{y} = \hat{y} - y$, which is exactly the signed error — positive when the network overestimates, negative when it underestimates.

4. **Gradient and backpropagation.** The gradient tells each weight how much its current value contributed to the loss, and in which direction to change. Backpropagation applies the chain rule layer by layer: the output-layer gradient flows back through $W^{(1,2)}$ and is elementwise-multiplied by $h_j(1-h_j)$ to produce the hidden-layer gradient. This means the network not only corrects the output weights, but also adjusts the hidden-layer weights that shaped the representation the output received.

5. **Parameter update and improvement.** Each weight moves in the direction that reduces the loss: $w \leftarrow w - \alpha \cdot \partial E / \partial w$. In every epoch of this activity, the prediction $\hat{y}$ increases slightly (moving toward $y = 1$) and the loss decreases. The improvement is incremental but consistent — which is the defining property of gradient descent on a smooth loss surface.

The phrase "learning from mistakes" is therefore precise: the loss encodes the mistake, the gradient distributes the blame across all weights proportionally to their contribution, and the update rule applies a proportional correction. After 15 epochs, the network has not yet solved the problem — but it is measurably better than it was at the start.

---
## Section 8 — Full XOR Table

We train on the complete XOR dataset:
$$X = \begin{pmatrix}0&0\\0&1\\1&0\\1&1\end{pmatrix}, \qquad y = \begin{pmatrix}0\\1\\1\\0\end{pmatrix}$$

Training runs for **5,000 epochs** with $\alpha = 1.0$ to allow full convergence. Weights are randomly initialized with `np.random.seed(42)` for reproducibility.

In [ ]:
# ── Full XOR — 5,000 epochs ───────────────────────────────────────────────────
np.random.seed(42)

X_xor = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
Y_xor = np.array([[0.],[1.],[1.],[0.]])

W01x = np.random.randn(2,2) * 0.8
W12x = np.random.randn(2,1) * 0.8
b1x  = np.zeros((1,2))
b2x  = np.zeros((1,1))
alpha_x = 1.0
loss_hist = []

for ep in range(5000):
    Z1x    = X_xor @ W01x + b1x
    H1x    = sigmoid(Z1x)
    Z2x    = H1x @ W12x + b2x
    Yhatx  = sigmoid(Z2x)
    loss_hist.append(float(np.mean(0.5*(Y_xor-Yhatx)**2)))

    d2x    = (Yhatx - Y_xor) * Yhatx * (1-Yhatx)
    dW12x  = H1x.T @ d2x
    db2x   = d2x.mean(axis=0, keepdims=True)
    d1x    = (d2x @ W12x.T) * H1x * (1-H1x)
    dW01x  = X_xor.T @ d1x
    db1x   = d1x.mean(axis=0, keepdims=True)

    W01x -= alpha_x*dW01x;  W12x -= alpha_x*dW12x
    b1x  -= alpha_x*db1x;   b2x  -= alpha_x*db2x

# Final predictions
Z1x    = X_xor @ W01x + b1x
H1x    = sigmoid(Z1x)
Yhatx  = sigmoid(H1x @ W12x + b2x)
preds  = (Yhatx >= 0.5).astype(int)

print("Final predictions after 5,000 epochs")
print("=" * 52)
print(f"{'x1':>4} {'x2':>4} {'y':>4} {'ŷ (prob)':>12} {'ŷ (class)':>11} {'Correct?':>10}")
print("-" * 52)
correct = 0
for i in range(4):
    xi1, xi2 = int(X_xor[i,0]), int(X_xor[i,1])
    yi   = int(Y_xor[i,0])
    yhi  = float(Yhatx[i,0])
    pi   = int(preds[i,0])
    ok   = "✓" if pi==yi else "✗"
    if pi==yi: correct += 1
    print(f"  {xi1:>2}   {xi2:>2}   {yi:>2}   {yhi:>10.5f}   {pi:>9}   {ok:>8}")
print("-" * 52)
print(f"  Accuracy: {correct}/4 = {correct/4*100:.0f}%")
print(f"  Final mean loss: {loss_hist[-1]:.6f}")

In [ ]:
# ── Full XOR plots ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Full XOR Training — 4 observations, 5,000 epochs, $\\alpha=1.0$",
             fontsize=12, fontweight="bold")

# Loss curve (log scale)
ax = axes[0]
ax.plot(np.arange(1,5001), loss_hist, color="#C8102E", lw=1.2)
ax.set_yscale("log")
ax.set_title("(a) Loss curve (log scale)", fontsize=11)
ax.set_xlabel("Epoch"); ax.set_ylabel("Mean loss $E$ (log scale)")
ax.grid(alpha=0.3)
ax.annotate(f"Start: {loss_hist[0]:.4f}",
            xy=(1, loss_hist[0]), xytext=(300, loss_hist[0]*1.8),
            fontsize=8, color="#C8102E")
ax.annotate(f"End: {loss_hist[-1]:.6f}",
            xy=(5000, loss_hist[-1]), xytext=(3500, loss_hist[-1]*3),
            fontsize=8, color="#C8102E")

# Decision boundary
ax2 = axes[1]
xx, yy = np.meshgrid(np.linspace(-0.1,1.1,200), np.linspace(-0.1,1.1,200))
grid = np.c_[xx.ravel(), yy.ravel()]
H1g  = sigmoid(grid @ W01x + b1x)
Pg   = sigmoid(H1g @ W12x + b2x).reshape(xx.shape)

ax2.contourf(xx, yy, Pg, levels=50, cmap="RdBu_r", alpha=0.7)
ax2.contour(xx, yy, Pg, levels=[0.5], colors=["black"], linewidths=[2])

c_map = {0:"#C8102E", 1:"#1a3e8a"}
labels_xor = {0:"Class 0", 1:"Class 1"}
plotted = set()
for i in range(4):
    yi = int(Y_xor[i,0])
    lbl = labels_xor[yi] if yi not in plotted else ""
    plotted.add(yi)
    ax2.scatter(X_xor[i,0], X_xor[i,1], s=220, c=c_map[yi],
                edgecolors="white", lw=2, zorder=5, label=lbl)
    ax2.annotate(f"({int(X_xor[i,0])},{int(X_xor[i,1])})  y={int(Y_xor[i,0])}",
                 (X_xor[i,0]+0.04, X_xor[i,1]+0.04), fontsize=8.5)

ax2.set_xlim(-0.1,1.2); ax2.set_ylim(-0.1,1.2)
ax2.set_title("(b) Learned decision boundary", fontsize=11)
ax2.set_xlabel(r"$x_1$"); ax2.set_ylabel(r"$x_2$")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

**Reflection.** The full XOR experiment confirms several theoretical claims.

**Convergence.** The loss drops from approximately 0.25 to near zero over 5,000 epochs. Panel (a) shows the characteristic *hockey stick* shape on the log scale: rapid initial descent followed by a long plateau, then a sharp drop as the network finds the right configuration. This behavior is typical of networks training on non-convex problems.

**Classification accuracy.** The model achieves 4/4 correct classifications with predicted probabilities well separated from the 0.5 threshold. This confirms that the network has genuinely learned the XOR function — not just memorized a pattern.

**Decision boundary.** Panel (b) shows the most important result: the learned boundary is **non-linear**, consisting of two curved regions. The red area corresponds to Class 0, the blue area to Class 1, and the black contour line is the decision boundary at $\hat{y} = 0.5$. No straight line can reproduce this geometry — which is exactly what the class notes argued in Figure 11.1 (Llinás, 2026). The hidden layer's neurons have implicitly defined two soft halfplanes, and the output neuron has combined them into the checkered XOR pattern.

**Key lesson.** The XOR problem could not be solved with one or two observations in 15 epochs. It required all four observations, more epochs, and a larger learning rate to converge. This illustrates a fundamental principle: a neural network can only learn the structure it has been shown. The training set is not just data — it is the totality of evidence the network can use to adjust its parameters.

---
## Reproducibility

In [ ]:
import sys
import matplotlib

df_v = pd.DataFrame([
    ("Python",     sys.version.split()[0],  "Base language"),
    ("numpy",      np.__version__,           "Vector/matrix operations, sigmoid"),
    ("pandas",     pd.__version__,           "DataFrames and tables"),
    ("matplotlib", matplotlib.__version__,   "Network diagram and training plots"),
], columns=["Library", "Version", "Role"])
df_v.index = range(1, len(df_v)+1)
print(df_v.to_string())
print()
print("Random seed          : np.random.seed(42)")
print("ML/autodiff libraries: none — all computations implemented from scratch")
print()
print("Reference:")
print("  Llinás, H. (2026). AI Materials — Table of Contents.")
print("  RPubs. https://rpubs.com/hllinas/R_AI1_toc")